# Targeted Iterative Fast Gradient Sign Method (I-FGSM) Attack

This notebook demonstrates the generation of targeted adversarial examples using the Iterative FGSM (I-FGSM) attack against pre-trained CNNs.

Unlike standard, untargeted attacks that maximize the loss of the true class to cause any misclassification, a **targeted** attack minimizes the loss with respect to a specific, chosen target class. By iteratively stepping in the negative direction of the gradient of the target class loss, we force the network to confidently misclassify the image as the target category.

In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Import models
from tensorflow.keras.applications import EfficientNetB0, InceptionV3, MobileNetV2

# Import specific preprocessing and decoding functions
from tensorflow.keras.applications.efficientnet import preprocess_input as preprocess_eff, decode_predictions as decode_eff
from tensorflow.keras.applications.inception_v3 import preprocess_input as preprocess_inc, decode_predictions as decode_inc
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as preprocess_mob, decode_predictions as decode_mob

I0000 00:00:1773130677.083633 3931144 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773130677.108288 3931144 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773130677.684293 3931144 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


### Setup Models and Images
We define `clip_min`, `clip_max`, and `eps_scale`. The `eps_scale` is critical because the step size $\alpha$ must be scaled proportionally depending on whether the model expects inputs in the `[-1, 1]` range or the `[0, 255]` range.

In [2]:
# Helper functions for denormalization
def denormalize_eff(img):
    img = tf.clip_by_value(img, 0, 255)
    return img.numpy().squeeze().astype("uint8")

def denormalize_neg1_1(img):
    img = (img + 1.0) / 2.0 
    img = img * 255.0       
    img = tf.clip_by_value(img, 0, 255)
    return img.numpy().squeeze().astype("uint8")

# Master dictionary with model configurations
models_dict = {
    'MobileNetV2': {
        'model': MobileNetV2(weights='imagenet'),
        'target_size': (224, 224),
        'preprocess_fn': preprocess_mob,
        'decode_fn': decode_mob,
        'denormalize_fn': denormalize_neg1_1,
        'clip_min': -1.0,
        'clip_max': 1.0,
        'eps_scale': 1.0
    },
    'EfficientNetB0': {
        'model': EfficientNetB0(weights='imagenet'),
        'target_size': (224, 224),
        'preprocess_fn': preprocess_eff,
        'decode_fn': decode_eff,
        'denormalize_fn': denormalize_eff,
        'clip_min': 0.0,
        'clip_max': 255.0,
        'eps_scale': 127.5 
    },
    'InceptionV3': {
        'model': InceptionV3(weights='imagenet'),
        'target_size': (299, 299),
        'preprocess_fn': preprocess_inc,
        'decode_fn': decode_inc,
        'denormalize_fn': denormalize_neg1_1,
        'clip_min': -1.0,
        'clip_max': 1.0,
        'eps_scale': 1.0
    },
}

for config in models_dict.values():
    config['model'].trainable = False

# Dictionary for target images
images_dict = {
    'Dog': '../images/dog.png',
    'Lion': '../images/lion.png'
}

InternalError: cudaSetDevice() on GPU:0 failed. Status: out of memory

In [ ]:
def load_and_preprocess_image(img_path, target_size, preprocess_fn):
    img_raw = tf.io.read_file(img_path)
    img = tf.image.decode_image(img_raw, channels=3)
    img = tf.cast(img, tf.float32)
    img = tf.image.resize(img, target_size)
    img = preprocess_fn(img)
    img = tf.expand_dims(img, axis=0)
    return img

def get_imagenet_label(prob, decode_fn):
    return decode_fn(prob, top=1)[0][0] 

def euclidean_distance(img1, img2):
    return np.linalg.norm(img1.numpy() - img2.numpy())

### Targeted I-FGSM Algorithm
To perform a targeted attack, the mathematical formulation is modified. Instead of adding the gradient to maximize the true class loss, we **subtract** the gradient to minimize the loss with respect to the target class $y_{target}$:
$$x_{adv}^{t+1} = \Pi_{x} \left( x_{adv}^t - \alpha \cdot sign(\nabla_x J(\theta, x_{adv}^t, y_{target})) \right)$$

The loop stops when the network predicts the target class with a confidence $\ge$ `target_confidence`, or when `max_iter` is reached.

In [ ]:
loss_object = tf.keras.losses.CategoricalCrossentropy()

def targeted_ifgsm_attack(input_image, target_label, current_model, clip_min, clip_max, eps_scale, base_alpha=0.01, target_confidence=0.95, max_iter=1000):
    
    # Scale alpha to the specific model's range
    alpha = base_alpha * eps_scale
    adv_image = tf.identity(input_image)
    
    # Get the index of the target class to track its probability
    target_class_idx = tf.cast(tf.argmax(target_label, axis=-1)[0], tf.int32)
    
    for i in range(max_iter):
        with tf.GradientTape() as tape:
            tape.watch(adv_image)
            prediction = current_model(adv_image)
            # Compute loss against the TARGET label, not the true label
            loss = loss_object(target_label, prediction)
            
        # We want to MINIMIZE this loss, so we get the gradients
        gradient = tape.gradient(loss, adv_image)
        signed_grad = tf.sign(gradient)
        
        # Take a step in the NEGATIVE direction of the gradient
        adv_image = adv_image - alpha * signed_grad
        
        # Project back to valid image ranges
        adv_image = tf.clip_by_value(adv_image, clip_min, clip_max)
        
        # Check current confidence for the target class
        current_preds = current_model(adv_image)
        current_conf = current_preds[0, target_class_idx]
        
        # Stop early if we reached the desired confidence
        if current_conf >= target_confidence:
            # print(f"Target reached at iteration {i} with confidence {current_conf:.4f}")
            break
            
    return adv_image

### Evaluation Loop
We choose a target class (e.g., `301` for Ladybug) and evaluate the attack across an array of increasing target confidences. This allows us to visualize how much perturbation is required to make the model progressively more certain of the wrong answer.

In [ ]:
# Array of target confidences we want to force the model to reach
target_confidences = [0.1, 0.3, 0.5, 0.7, 0.9, 0.99]
num_cols = len(target_confidences) + 1 

# The ImageNet index we want to target (e.g., 301 = ladybug, 340 = zebra)
target_class_index = 301 

for model_name, m_config in models_dict.items():
    print(f"\n{'='*80}")
    print(f"EXECUTING TARGETED I-FGSM ATTACKS ON MODEL: {model_name}")
    print(f"{'='*80}\n")
    
    current_model = m_config['model']
    target_size = m_config['target_size']
    preprocess_fn = m_config['preprocess_fn']
    decode_fn = m_config['decode_fn']
    denormalize_fn = m_config['denormalize_fn']
    
    clip_min = m_config['clip_min']
    clip_max = m_config['clip_max']
    eps_scale = m_config['eps_scale']
    
    for img_name, img_path in images_dict.items():
        print(f"--- Original Image: {img_name} | Target Class Index: {target_class_index} ---")
        
        fig, axes = plt.subplots(1, num_cols, figsize=(4 * num_cols, 4.5))
        fig.subplots_adjust(wspace=0.1)
        
        # 1. Load and Preprocess
        input_img = load_and_preprocess_image(img_path, target_size, preprocess_fn)
        
        # 2. Get Original Prediction
        original_probs = current_model.predict(input_img, verbose=0)
        _, orig_class_name, orig_prob = get_imagenet_label(original_probs, decode_fn)
        
        # Create One-Hot label for the TARGET class
        target_label = tf.one_hot(target_class_index, original_probs.shape[-1])
        target_label = tf.reshape(target_label, (1, original_probs.shape[-1]))
        
        # Plot Original Image
        ax_orig = axes[0]
        ax_orig.imshow(denormalize_fn(input_img)) 
        ax_orig.set_title(f'ORIGINAL\n{orig_class_name}\n({orig_prob * 100:.2f}%)', fontweight='bold')
        ax_orig.axis('off')
        
        # 3. Iterate over different target confidences
        for col_idx, conf_val in enumerate(target_confidences, start=1):
            
            # Execute Targeted I-FGSM Attack
            # base_alpha=0.01 is a small step. Feel free to adjust.
            adv_img = targeted_ifgsm_attack(input_img, target_label, current_model, clip_min, clip_max, eps_scale, base_alpha=0.01, target_confidence=conf_val, max_iter=1000) 
            
            # Calculate L2 Distance
            l2 = euclidean_distance(input_img, adv_img)
            
            # Get Adversarial Prediction
            adv_img_probs = current_model.predict(adv_img, verbose=0)
            _, adv_img_class, adv_class_prob = get_imagenet_label(adv_img_probs, decode_fn)
            
            # Plot Adversarial Image
            ax = axes[col_idx]
            ax.imshow(denormalize_fn(adv_img)) 
            ax.set_title(f'Target Conf $\ge$ {conf_val}\n{adv_img_class}\n({adv_class_prob * 100:.2f}%)')
            ax.text(0.5, -0.15, f'$L_2={l2:.2f}$', transform=ax.transAxes, ha='center', fontsize=10)
            ax.axis('off')
            
        plt.tight_layout()
        plt.show()